In [ ]:
# Run this cell. Use this folder's Python env so `datascience` is available:
#   cd visualizations-workshop && python3 -m venv .venv && .venv/bin/pip install -r requirements.txt
# Then pick kernel/interpreter: visualizations-workshop/.venv/bin/python
from datascience import *
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline
plt.style.use("ggplot")
import warnings
warnings.filterwarnings('ignore')

<img src="https://github.com/data-6-berkeley/materials-fa24/blob/main/hw/hw03/data6.png?raw=true" style="width: 15%; float: right; padding: 1%; margin-right: 2%;"/>

# Bickel Case Study

**Goals:**
- Load and inspect real admissions data  
- Compare overall and department-level admission rates by gender  
- Visualize Simpson’s Paradox  

**Dataset Source:** The [dataset](https://discovery.cs.illinois.edu/dataset/berkeley/) used in this notebook is a cleaned version of the original UC Berkeley 1973 graduate admissions data, processed by the University of Illinois at Urbana-Champaign. 

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 1973 UC Berkeley Graduate Admissions

<hr style="border: 1px solid #fdb515;" />

## Table Methods You'll Use

This notebook uses a small set of `datascience` `Table` methods repeatedly. Each topic below is followed by a **runnable** cell that uses `UCBerkeley1973_Admission.csv` (loaded in the first code cell).

In [ ]:
# `read_table`, `show`, `num_rows` — Berkeley admissions CSV (same file as below)
cal_data = Table.read_table("UCBerkeley1973_Admission.csv")
cal_data.show(5)
cal_data.num_rows, cal_data.num_columns

## The `where` method

`where` keeps only rows that match a condition.

- Exact match: `tbl.where("Gender", "F")`
- You can chain filters to narrow the table step by step.

Example from this notebook:

- `cal_data.where("Admission", "Accepted").where("Gender", "F")`
- Adding `.num_rows` counts how many rows remain after filtering.

This is how the notebook computes accepted applicants by gender.

In [ ]:
# `where` — chain filters; `.num_rows` counts matching rows
cal_data.where("Admission", "Accepted").where("Gender", "F").num_rows, cal_data.where("Admission", "Accepted").where("Gender", "M").num_rows

## `pivot`, `with_column`, and `make_array`

The [`pivot`](http://data8.org/datascience/_autosummary/datascience.tables.Table.pivot.html#datascience.tables.Table.pivot) method summarizes combinations of two categorical columns.

- `pivot(columns, rows)` counts how many rows fall in each row-column pair.
- With `values=` and `collect=`, each cell stores an aggregated value instead of a count.

Examples from this notebook:

- `cal_data.pivot("Admission", "Gender")` — each row is a gender; columns are `Accepted` and `Rejected` counts
- `cal_data.pivot("Gender", "Major", values="Admission", collect=...)` — each row is a major; columns are `F` and `M` rates

`with_column` adds one new column to a table.

- `tbl.with_column("New Label", values)`

`make_array(...)` creates an array of values, which is useful when manually adding a small column, like the two overall acceptance rates.

In [ ]:
# `pivot` — rows = Gender; columns = Admission outcome (counts in each cell)
admission_by_gender = cal_data.pivot("Admission", "Gender")
admission_by_gender
# `with_column` — one value per row (add the two admission columns = applicants per gender)
admission_by_gender.with_column(
    "applicants_per_gender",
    admission_by_gender.column("Accepted") + admission_by_gender.column("Rejected"),
)
# `pivot` with `values` and `collect` (same pattern as department rates later)
cal_data.pivot(
    "Gender",
    "Major",
    values="Admission",
    collect=...,
)
# `make_array` — overall acceptance rate for F and for M
make_array(
    cal_data.where("Gender", "F").where("Admission", "Accepted").num_rows / cal_data.where("Gender", "F").num_rows,
    cal_data.where("Gender", "M").where("Admission", "Accepted").num_rows / cal_data.where("Gender", "M").num_rows,
)

## `group`, `relabeled`, `bar`, and `barh`

The [`group`](http://data8.org/datascience/_autosummary/datascience.tables.Table.group.html#datascience.tables.Table.group) method collects rows into bins.

- `tbl.group("Column")` counts rows in each category.
- In this notebook, `group([ ... ])` is used to prepare totals before plotting Simpson's Paradox.

`relabeled` renames columns in a copy of a table.

- `tbl.relabeled(["old1", "old2"], ["new1", "new2"])`

Bar charts in `datascience` work best on small summary tables:

- `tbl.bar("Category")` makes a vertical bar chart.
- `tbl.barh("Category")` makes a horizontal bar chart.
- `overlay=True` draws multiple series on the same axes.

Examples used later:

- `admissions.barh("Gender", "Acceptance Rate")`
- `admission_major.bar("Major", overlay=True)`

In [ ]:
# `group`, `relabeled`, `barh` — plot a small aggregated table
by_major = cal_data.group("Major")
by_major
by_major.relabeled("count", "applicants").barh("Major", "applicants")
# `bar` with `overlay=True` — two series (same idea as department rates later)
rates_by_major = cal_data.pivot(
    "Gender",
    "Major",
    values="Admission",
    collect=lambda apps: sum(apps == "Accepted") / len(apps),
)
rates_by_major.bar("Major", overlay=True)

In [ ]:
# Load data from CSV
cal_data = Table.read_table('UCBerkeley1973_Admission.csv')
cal_data.show(5)

In [ ]:
total_f = sum(...)
total_m = sum(...)
accepted_f = cal_data.where("Admission", "Accepted").where(..., ...).num_rows
accepted_m =  cal_data.where("Admission", "Accepted").where(..., ...).num_rows
acceptance_rate_f = accepted_f / total_f 
acceptance_rate_m = accepted_m / total_m
print("1973's Berkeley admission rate seems to be: female:", acceptance_rate_f * 100, "and male:", acceptance_rate_m * 100)

<hr style="border: 1px solid #fdb515;" />

## Data Talk

<div class="alert alert-info">

1. **What do you notice?**

2. **What do you wonder?**

3. **What story does this tell, especially about the community this data may impact?**

</div>

| Group   | Applicants | Admitted | Men Applicants | Men Admitted                                | Women Applicants | Women Admitted |
|---------|------------|----------|----------------|----------------------------------------------|------------------|----------------|
| **Total** | 12,763 | 41%   | 8,442       | <span style="color:green"><b>44%</b></span> | 4,321         | **35%**          |

| Dept | All Applicants | All Admitted | Men Applicants | Men Admitted | Women Applicants | Women Admitted |
|------|----------------|--------------|----------------|--------------|------------------|----------------|
| A    | 933            | 64%          | <span style="color:blue"><b>**825**</b></span> | 62%          | 108              | <span style="color:green"><b>82%</b></span> |
| B    | 585            | 63%          | <span style="color:blue"><b>**560**</b></span> | 63%          | 25               | <span style="color:green"><b>68%</b></span> |
| C    | 918            | 35%          | 325            | <span style="color:green"><b>37%</b></span> | <span style="color:blue"><b>**593**</b></span> | 34% |
| D    | 792            | 34%          | <span style="color:blue"><b>417</b></span> | 33%          | 375              | 35% |
| E    | 584            | 25%          | 191            | <span style="color:green"><b>28%</b></span> | <span style="color:blue"><b>**393**</b></span> | 24% |
| F    | 714            | 6%           | <span style="color:blue"><b>373</b></span> | 6%           | 341              | <span style="color:green"><b>7%</b></span> |
| **Total** | **4,526**      | **39%**       | **2,691**      | **45%**       | **1,835**        | **30%**        |

<hr style="border: 1px solid #fdb515;" />

## Visualization Talk

<div class="alert alert-info">

1. **What do you notice?**

2. **What do you wonder?**

3. **What columns plotted what?** (Focus on axes)

</div>

In [ ]:
admissions = cal_data.pivot('Admission', 'Gender').with_column("Acceptance Rate", make_array(acceptance_rate_f, acceptance_rate_m))
admissions.barh("Gender", "Acceptance Rate")

In [ ]:
admission_major = cal_data.pivot('Gender', 'Major', collect = lambda x: sum(x == 'Accepted') / len(x), values = 'Admission')
admission_major = admission_major.relabeled(['F', 'M'], ['F Acceptance Rate', 'M Acceptance Rate'])
admission_major.bar('Major', overlay = True)

<hr style="border: 1px solid #fdb515;" />

## Simpson’s Paradox Coding

Instead of visualizing admission rates, let’s focus on the total number of admitted applicants. Modify the code to produce a bar chart as shown below:

<img src="https://github.com/dubois-ctds/data6-nwdse-2025/blob/main/visualizations-workshop/barchart.png?raw=true" alt="Bar Chart"/>

In [ ]:
num_applicants = cal_data.group([_____])._____('Gender', 'Major', collect = _____, values = 'count')
num_applicants = num_applicants.relabeled(['F', 'M'], ['F Application Count', 'M Application Count'])
num_applicants._____(_____) # barchart